# 01_chech_excel_cols
------
- Blob Storage에서 7개 부두 엑셀 파일 목록 확인 성공
- `.xlsx` 파일은 `openpyxl`로 읽기 가능
- `.xls` 파일 일부는 실제 Excel binary가 아니라 HTML table 형식으로 확인됨
- 따라서 파일 확장자에 따라 읽기 방식을 분기해야 함
  - `.xlsx`: `pd.read_excel(engine="openpyxl")`
  - `.xls`: `pd.read_html()`
- 부두별 컬럼 개수와 헤더 위치가 서로 다름
  - 1, 2, 4부두: header 없음 → 컬럼명 수동 부여 완료
  - 3,5,6,7부두: 엑셀 내부 헤더 존재
--------


### 0. 패키지 및 라이브러리 설치

In [0]:
%pip install fsspec openpyxl

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 3.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install xlrd

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 2.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install html5lib lxml beautifulsoup4

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 2.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

### 1. blob 에서 로드하기

In [0]:
# ==================
# 1: blob 토큰으로 연결
# ==================
storage_account = "4dtteam3storage"
container = "demo-raw" 
sas_token = "<TOKEN>"  

spark.conf.set(
    f"fs.azure.sas.{container}.{storage_account}.blob.core.windows.net",
    sas_token
)

In [0]:
# ===========
# 2: 경로 확인
# ===========
blob_path = f"wasbs://{container}@{storage_account}.blob.core.windows.net/"

files = dbutils.fs.ls(blob_path)
for f in files:
    print(f.name, f.size)

terminal_1_demo_0624.xls 10321
terminal_2_demo_0624.xls 6494
terminal_3_demo_0624.xlsx 4808
terminal_4_demo_0624.xls 5692
terminal_5_demo_0624.xlsx 18743
terminal_6_demo_0624.xlsx 5488
terminal_7_demo_0624.xlsx 20333


In [0]:
# =============
# 3: Blob 엑셀을 로컬로 복사하는 함수
# =============
import os
import json
from openpyxl import load_workbook

def copy_blob_file_to_local(blob_file_path):
    workspace_tmp_dir = "/Workspace/Users/4dt035@msacademy.msai.kr/tmp_excel"
    dbutils.fs.mkdirs(workspace_tmp_dir)
    
    local_path = f"{workspace_tmp_dir}/{os.path.basename(blob_file_path)}"
    dbutils.fs.cp(blob_file_path, f"file:{local_path}")

    return local_path

In [0]:
# ===========
# 4: 확장자별 pandas engine 선택
# ===========
def get_excel_engine(file_path):
    if file_path.endswith(".xlsx"):
        return "openpyxl"
    elif file_path.endswith(".xls"):
        return "xlrd"
    else:
        return None

In [0]:
import os
import pandas as pd

# ===========
# 5: 파일 읽기 함수
# ===========
def read_excel_or_html_table(local_path, nrows=10):
    file_name = os.path.basename(local_path)

    # xlsx는 진짜 엑셀이므로 openpyxl
    if file_name.endswith(".xlsx"):
        xls = pd.ExcelFile(local_path, engine="openpyxl")
        results = []

        for sheet in xls.sheet_names:
            raw = pd.read_excel(
                local_path,
                sheet_name=sheet,
                header=None,
                nrows=nrows,
                engine="openpyxl"
            )

            results.append({
                "sheet": sheet,
                "shape_preview": raw.shape,
                "first_5_rows": raw.head(5).values.tolist()
            })

        return results

    # xls는 HTML 표일 가능성이 있으므로 read_html
    elif file_name.endswith(".xls"):
        tables = read_html_xls_safely(local_path)
        results = []

        for i, table in enumerate(tables):
            raw = table.head(nrows)

            results.append({
                "sheet": f"html_table_{i}",
                "shape_preview": raw.shape,
                "first_5_rows": raw.head(5).values.tolist()
            })

        return results

In [0]:
# ===========
# 6: 판다스로 바로 읽기
# 3,5,6,7 부두 성공
# 1,2,4 부두 실패 후 성공: xls 파일이 아닌 html이었음. 
# ===========
import pandas as pd

# 1단계에서 리스팅한 파일 경로들을 리스트로
file_paths = [f.path for f in files if f.name.endswith(('.xlsx', '.xls'))]

results = []

for path in file_paths:
    try:
        local_path = copy_blob_file_to_local(path)
        table_results = read_excel_or_html_table(local_path, nrows=10)

        for table_result in table_results:
            results.append({
                "file": os.path.basename(path),
                "sheet": table_result["sheet"],
                "shape_preview": table_result["shape_preview"],
                "first_5_rows": table_result["first_5_rows"]
            })

    except Exception as e:
        results.append({
            "file": os.path.basename(path),
            "error": str(e)
        })

for r in results:
    print(r)
    print("-" * 80)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-61b88033-4cba-4106-b592-d944dee87d62/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-61b88033-4cba-4106-b592-d944dee87d62/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{'file': 'terminal_1_demo_0624.xls', 'sheet': 'html_table_0', 'shape_preview': (4, 15), 'first_5_rows': [['T3(P)', 'HLC', 'IQUX004', '2623E/2623E', '41 (49) 62', 'IQUIQUE EXPRESS', 'AN3E', '2026-06-21 18:00', '2026-06-22 05:30', '2026-06-24 11:12', 2151, 2991, 278, 'N', 'DEPARTED'], ['T1(P)', 'ZIM', 'ZSEM004', '6S/6S', '01 (13) 18', 'SEASMILE', 'PANDA', '2026-06-22 00:00', '2026-06-22 10:25', '2026-06-24 08:00', 1316, 1276, 322, 'N', 'DEPARTED'], ['T1(P)', 'MSC', 'MABX001', 'UK623A/UK623A', '02 (16) 23', 'MSC ABY X', 'CHINKE', '2026-06-24 00:00', '2026-06-24 10:45', '2026-06-25 23:00', 981, 1064, 346, 'N', 'ARRIVED'], ['T3(P)', 'MSC', 'MREG002', 'UX623A/UX623A', '41 (50) 63', 'MSC REGULUS', 'SANTA', '2026-06-24 03:00', '2026-06-24 14:00', '2026-06-26 13:00', 1096, 2348, 818, 'N', 'ARRIVED']]}
--------------------------------------------------------------------------------
{'file': 'terminal_2_demo_0624.xls', 'sheet': 'html_table_0', 'shape_preview': (5, 17), 'first_5_rows': [[1, 'MAERS

In [0]:
# ===============
# 1,2,4 부두 실패 원인 : html 이었음.
# ===============

local_path = copy_blob_file_to_local("wasbs://demo-raw@4dtteam3storage.blob.core.windows.net/terminal_1_demo_0624.xls")

with open(local_path, "r", encoding="utf-8", errors="ignore") as f:
    print(f.read(500))














 











<script type="text/javascript" src="../common/js/jquery-3.7.1.min.js"></script>
<script type="text/javascript" src="../common/js/jquery-ui.min.js"></script>


<script language="JavaScript" type="text/javascript" nonce="bgisnikaec">
	$(document).ready(function() {
		$('form').each(function() {
			$(this).append($('<input/>', {type: 'hidden', name: 'CSRF_TOKEN', value:'66fea814-29e9-4f2f-9a66-b0cd43054b31' }));
		})
	})
</script>
 







<script language="JavaScript" typ


In [0]:
# =========
# 파일목록정보
# =========
summary_rows = []

for r in results:
    if "error" in r:
        summary_rows.append({
            "file": r["file"],
            "sheet": None,
            "row_count_preview": None,
            "column_count_preview": None,
            "status": "ERROR",
            "error": r["error"]
        })
    else:
        summary_rows.append({
            "file": r["file"],
            "sheet": r["sheet"],
            "row_count_preview": r["shape_preview"][0],
            "column_count_preview": r["shape_preview"][1],
            "status": "SUCCESS",
            "error": None
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

file,sheet,row_count_preview,column_count_preview,status,error
terminal_1_demo_0624.xls,html_table_0,4,15,SUCCESS,null
terminal_2_demo_0624.xls,html_table_0,5,17,SUCCESS,null
terminal_3_demo_0624.xlsx,Berth_Schedule,8,18,SUCCESS,null
terminal_4_demo_0624.xls,html_table_0,3,16,SUCCESS,null
terminal_5_demo_0624.xlsx,Sheet1,4,9,SUCCESS,null
terminal_6_demo_0624.xlsx,sheet1,8,17,SUCCESS,null
terminal_7_demo_0624.xlsx,Sheet1,6,12,SUCCESS,null


### 2. 1,2,4부두 header 없음 -> 컬럼명 수동 부여

In [0]:
print(local_path)

/Workspace/Users/4dt035@msacademy.msai.kr/tmp_excel/terminal_7_demo_0624.xlsx


In [0]:
# ===================
# 1,2,4부두 .xls(html) 전체 임시컬럼 붙이기
# ====================
import os
import pandas as pd
from io import StringIO

xls_paths = [f.path for f in files if f.name.endswith(".xls")]

html_results = []

for path in xls_paths:
    local_path = copy_blob_file_to_local(path)

    with open(local_path, "rb") as f:
        content = f.read()

    content = content.replace(b"udf-8", b"utf-8").replace(b"UDF-8", b"UTF-8")

    try:
        html_text = content.decode("utf-8")
    except UnicodeDecodeError:
        html_text = content.decode("cp949", errors="replace")

    tables = pd.read_html(StringIO(html_text))
    df = tables[0]
    df.columns = [f"col_{i+1}" for i in range(df.shape[1])]

    html_results.append({
        "file": os.path.basename(path),
        "rows": df.shape[0],
        "cols": df.shape[1],
        "columns": df.columns.tolist()
    })

    print(os.path.basename(path))
    display(df.head())
    print(df.shape)
    print("-" * 80)

summary_html_df = pd.DataFrame(html_results)
display(summary_html_df)

terminal_1_demo_0624.xls


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15
T3(P),HLC,IQUX004,2623E/2623E,41 (49) 62,IQUIQUE EXPRESS,AN3E,2026-06-21 18:00,2026-06-22 05:30,2026-06-24 11:12,2151,2991,278,N,DEPARTED
T1(P),ZIM,ZSEM004,6S/6S,01 (13) 18,SEASMILE,PANDA,2026-06-22 00:00,2026-06-22 10:25,2026-06-24 08:00,1316,1276,322,N,DEPARTED
T1(P),MSC,MABX001,UK623A/UK623A,02 (16) 23,MSC ABY X,CHINKE,2026-06-24 00:00,2026-06-24 10:45,2026-06-25 23:00,981,1064,346,N,ARRIVED
T3(P),MSC,MREG002,UX623A/UX623A,41 (50) 63,MSC REGULUS,SANTA,2026-06-24 03:00,2026-06-24 14:00,2026-06-26 13:00,1096,2348,818,N,ARRIVED


(4, 15)
--------------------------------------------------------------------------------
terminal_2_demo_0624.xls


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17
1,MAERSK NOTODDEN,NTOD-004/2026,624N/625S,MCC,HP2,Star,2026-06-23 12:25,2026-06-24 06:45,B6,2026-06-23 00:00,405,757,2,ENS MARINE,null,2026-06-23 08:02
2,ESL SANA,EMSN-002/2026,26B7W/26B7W,ESL,GLX,Port,2026-06-23 22:20,2026-06-24 10:00,B7,2026-06-23 11:00,23,588,32,New Port Marine,null,2026-06-23 18:02
3,JPO LIBRA,MJBR-007/2026,625S/626N,MAE,A04,Port,2026-06-24 00:45,2026-06-25 10:00,B5,2026-06-23 13:00,1899,996,0,ENS MARINE,null,2026-06-23 23:02
4,GSL GRANIA,GGRN-004/2026,624E/624E,MAE,GUSWC3E,Port,2026-06-24 08:25,2026-06-25 15:00,B6,2026-06-23 21:00,1016,1026,466,ENS MARINE,null,2026-06-24 04:02
5,TEMA MAERSK,TEMM-001/2026,624E/624E,MAE,GUSEC3,Port,2026-06-24 15:15,2026-06-26 02:00,B7,2026-06-24 00:00,1209,2493,34,ENS MARINE,null,2026-06-24 08:02


(5, 17)
--------------------------------------------------------------------------------
terminal_4_demo_0624.xls


col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,col_10,col_11,col_12,col_13,col_14,col_15,col_16
T2(S),ONE,OOST001,029E/029E,ONE STORK,EC2E,2026-06-19 00:00,2026-06-21 20:30,2026-06-22 08:30,2026-06-24 22:00,2026-06-19 01:00,1297,4061,350,N,DEPARTED
T1(S),ONE,O1DN012,0007N/0007N,ONE DANIELLA,JPHN,2026-06-20 00:00,2026-06-23 06:06,2026-06-23 18:06,2026-06-24 07:00,2026-06-21 04:00,200,238,0,N,DEPARTED
T1(S),HMM,HHMN006,0025W/0025W,HMM MANILA,AAD,2026-06-21 00:00,2026-06-23 20:30,2026-06-24 08:30,2026-06-25 00:00,2026-06-22 11:00,337,222,22,N,DEPARTED


(3, 16)
--------------------------------------------------------------------------------


file,rows,cols,columns
terminal_1_demo_0624.xls,4,15,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15)"
terminal_2_demo_0624.xls,5,17,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16, col_17)"
terminal_4_demo_0624.xls,3,16,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16)"


### 3. 진단표

In [0]:
# ===========
# 부두별 파일 구조 진단표 만들기
# ===========

import os
import pandas as pd
from io import StringIO

# 헤더 행을 찾을 때 기준으로 사용할 선석 스케줄 주요 컬럼 키워드 목록
HEADER_KEYWORDS = [
    "No", "선석", "항로", "모선항차", "선박명", "선사항차",
    "선사", "ETA", "ETD", "양하", "적하", "Shift", "상태", "AMP"
]

def read_html_xls(local_path):
    '''
    .xls 확장자로 저장된 HTML 표 파일을 pandas DataFrame으로 읽어오는 함수
    '''
    with open(local_path, "rb") as f:
        content = f.read()

    content = content.replace(b"udf-8", b"utf-8").replace(b"UDF-8", b"UTF-8")

    try:
        html_text = content.decode("utf-8")
    except UnicodeDecodeError:
        html_text = content.decode("cp949", errors="replace")

    tables = pd.read_html(StringIO(html_text))
    return tables[0]


def detect_header_row(preview_df):
    '''
    상단 미리보기 데이터에서 컬럼명으로 보이는 헤더 행을 자동으로 찾는 함수
    '''
    best_row = None
    best_score = 0

    for idx, row in preview_df.iterrows():
        values = [str(v).strip() for v in row.tolist() if pd.notna(v)]
        score = sum(
            1 for v in values
            for keyword in HEADER_KEYWORDS
            if keyword.lower() in v.lower()
        )

        if score > best_score:
            best_score = score
            best_row = idx

    if best_score == 0:
        return None, "no_header"

    return best_row, f"header_row_{best_row + 1}"


diagnosis_rows = []

file_paths = [
    f.path for f in files
    if f.name.endswith((".xlsx", ".xls"))
]

# ---------------
# Blob에 있는 7개 부두 파일을 하나씩 읽고, 파일 형식/시트명/컬럼 수/헤더 위치를 진단표로 정리하는 코드
# ---------------
for path in file_paths:
    file_name = os.path.basename(path)
    local_path = copy_blob_file_to_local(path)

    try:
        if file_name.endswith(".xls"):
            df = read_html_xls(local_path)

            diagnosis_rows.append({
                "file": file_name,
                "read_type": "html_xls",
                "sheet_or_table": "html_table_0",
                "row_count": df.shape[0],
                "column_count": df.shape[1],
                "header_row": None,
                "header_status": "no_header_or_manual_check",
                "need_manual_columns": "Y",
                "columns_preview": [f"col_{i+1}" for i in range(df.shape[1])],
                "first_row_preview": df.head(1).values.tolist()
            })

        elif file_name.endswith(".xlsx"):
            xls = pd.ExcelFile(local_path, engine="openpyxl")

            for sheet in xls.sheet_names:
                preview_df = pd.read_excel(
                    local_path,
                    sheet_name=sheet,
                    header=None,
                    nrows=10,
                    engine="openpyxl"
                )

                header_row, header_status = detect_header_row(preview_df)

                if header_row is not None:
                    df = pd.read_excel(
                        local_path,
                        sheet_name=sheet,
                        header=header_row,
                        engine="openpyxl"
                    )
                    need_manual = "N"
                    cols_preview = df.columns.tolist()
                else:
                    df = preview_df
                    need_manual = "Y"
                    cols_preview = [f"col_{i+1}" for i in range(df.shape[1])]

                diagnosis_rows.append({
                    "file": file_name,
                    "read_type": "xlsx",
                    "sheet_or_table": sheet,
                    "row_count": df.shape[0],
                    "column_count": df.shape[1],
                    "header_row": None if header_row is None else header_row + 1,
                    "header_status": header_status,
                    "need_manual_columns": need_manual,
                    "columns_preview": cols_preview,
                    "first_row_preview": preview_df.head(1).values.tolist()
                })

    except Exception as e:
        diagnosis_rows.append({
            "file": file_name,
            "read_type": None,
            "sheet_or_table": None,
            "row_count": None,
            "column_count": None,
            "header_row": None,
            "header_status": "ERROR",
            "need_manual_columns": None,
            "columns_preview": None,
            "first_row_preview": None,
            "error": str(e)
        })

diagnosis_df = pd.DataFrame(diagnosis_rows)
diagnosis_df['first_row_preview'] = diagnosis_df['first_row_preview'].astype(str)

display(diagnosis_df)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-61b88033-4cba-4106-b592-d944dee87d62/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-61b88033-4cba-4106-b592-d944dee87d62/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-61b88033-4cba-4106-b592-d944dee87d62/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/local_disk0/.ephemeral_nfs/envs/pythonEnv-61b88033-4cba-4106-b592-d944dee87d62/lib/python3.10/site-packages/openpyxl/styles/styleshe

file,read_type,sheet_or_table,row_count,column_count,header_row,header_status,need_manual_columns,columns_preview,first_row_preview
terminal_1_demo_0624.xls,html_xls,html_table_0,4,15,null,no_header_or_manual_check,Y,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15)","[['T3(P)', 'HLC', 'IQUX004', '2623E/2623E', '41 (49) 62', 'IQUIQUE EXPRESS', 'AN3E', '2026-06-21 18:00', '2026-06-22 05:30', '2026-06-24 11:12', 2151, 2991, 278, 'N', 'DEPARTED']]"
terminal_2_demo_0624.xls,html_xls,html_table_0,5,17,null,no_header_or_manual_check,Y,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16, col_17)","[[1, 'MAERSK NOTODDEN', 'NTOD-004/2026', '624N/625S', 'MCC', 'HP2', 'Star', '2026-06-23 12:25', '2026-06-24 06:45', 'B6', '2026-06-23 00:00', 405, 757, 2, 'ENS MARINE', nan, '2026-06-23 08:02']]"
terminal_3_demo_0624.xlsx,xlsx,Berth_Schedule,6,18,2.0,header_row_2,N,"List(No, 선석, 항로, 모선항차, 선박명, 선사항차, 접안, 선사, 반입 시작일시, 반입 마감일시, 입항일시, 출항일시, 작업 시작일시, 작업 완료일시, 양하, 선적, S/H, 전배)","[['Berth_Schedule', nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]]"
terminal_4_demo_0624.xls,html_xls,html_table_0,3,16,null,no_header_or_manual_check,Y,"List(col_1, col_2, col_3, col_4, col_5, col_6, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16)","[['T2(S)', 'ONE', 'OOST001', '029E/029E', 'ONE STORK', 'EC2E', '2026-06-19 00:00', '2026-06-21 20:30', '2026-06-22 08:30', '2026-06-24 22:00', '2026-06-19 01:00', 1297, 4061, 350, 'N', 'DEPARTED']]"
terminal_5_demo_0624.xlsx,xlsx,Sheet1,3,9,1.0,header_row_1,N,"List(선석, 선사, 모선항차(선사항차) Head (Bridge) Stern, 선명 (ROUTE), 반입마감시한, 접안(예정)일시, 출항(예정)일시, 작업량 양하 / 적하 / Shift, 상태)","[['선석', '선사', '모선항차(선사항차)\nHead (Bridge) Stern', '선명\n(ROUTE)', '반입마감시한', '접안(예정)일시', '출항(예정)일시', '작업량\n양하 / 적하 / Shift', '상태']]"
terminal_6_demo_0624.xlsx,xlsx,sheet1,7,17,1.0,header_row_1,N,"List(No, 선석, 선사, 모선/항차, 입항, 출항, CCT, 접안예정시간(ETB), 출항예정시간(ETD), 양하, 적하, 이적, 모선명, ROUTE, 전배TML, 검역, 상태)","[['No', '선석', '선사', '모선/항차', '입항', '출항', 'CCT', '접안예정시간(ETB)', '출항예정시간(ETD)', '양하', '적하', '이적', '모선명', 'ROUTE', '전배TML', '검역', '상태']]"
terminal_7_demo_0624.xlsx,xlsx,Sheet1,5,12,1.0,header_row_1,N,"List(선석, 선사코드, 모선항차(선사항차), 모선명(Route), 반입마감시한, 접안예정일시, 출항예정일시, 작업시작시간, 작업완료시간, Head (Bridge) Stern, 작업량 양하/적하/Shift, 상태)","[['선석', '선사코드', '모선항차(선사항차)', '모선명(Route)', '반입마감시한', '접안예정일시', '출항예정일시', '작업시작시간', '작업완료시간', 'Head (Bridge) Stern', '작업량\n양하/적하/Shift', '상태']]"
